# 04. Limpieza de SISAM (calidad del aire)

**Propósito.** Limpiar y unificar los datos diarios de calidad del aire de SISAM (INPE y Fiocruz) por municipio: PM10, PM2.5, O3, NO2, CO y SO2. Para 2022 a 2025 se usa el **reanálisis**; para 2026 solo existe el **pronóstico**, así que se armonizan las columnas y se marca la fuente de cada fila.

**Entradas** (`SISAM/`): todos los `sisam_municipio_*.csv` crudos (`reanalise` 2022 a 2025 y `forecast` 2026-01-01 a 2026-10-31). Opcional: `BDQueimadas/centroides_municipio.csv` y `panel_semanal_base.csv` para verificar la cobertura de municipios.

**Salida:** `SISAM/sisam_limpio.csv`.

**Qué cambió respecto al Hito 2**
- Se leen todos los archivos por patrón y se agrega el pronóstico de 2026.
- Las columnas del pronóstico (`*_forecast_0`) se renombran con un diccionario explícito a las del reanálisis, con `assert` de que ambas fuentes quedan iguales.
- Nueva columna `fuente_sisam` (`reanalise` o `pronostico`) y `uf` por mapeo de estado.
- Se verifica que `(municipio_norm, uf, data)` no se repita; si el pronóstico trajera varias corridas por día, el notebook se detiene sin decidir cómo resolverlo.

In [1]:
from pathlib import Path
import unicodedata

import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# 1. Entorno: Google Colab (Drive) o ejecucion local
# ---------------------------------------------------------------------------
USAR_DRIVE = True   # True en Colab; False para trabajar con la carpeta local del repositorio
BASE_DRIVE = Path("/content/drive/MyDrive/BIODIVERSITY_GUARD")

if USAR_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        BASE = BASE_DRIVE
    except ImportError:
        print("No se detecto Google Colab: se usa la raiz local del repositorio.")
        USAR_DRIVE = False

if not USAR_DRIVE:
    # la raiz es la primera carpeta (hacia arriba desde la actual) que contiene "datos"
    _aqui = Path.cwd().resolve()
    BASE = next((p for p in [_aqui, *_aqui.parents] if (p / "datos").is_dir()), None)
    if BASE is None:
        raise FileNotFoundError("No se encontro la carpeta 'datos' subiendo desde " + str(_aqui))

if not BASE.exists():
    raise FileNotFoundError(f"No existe BASE = {BASE}. Verifica la carpeta BIODIVERSITY_GUARD en Drive.")

# ---------------------------------------------------------------------------
# 2. Rutas (todas salen de BASE)
# ---------------------------------------------------------------------------
DIR_PREP = BASE / "datos" / "datasets_preprocesamiento"
DIR_BDQ = DIR_PREP / "BDQueimadas"
DIR_DETER = DIR_PREP / "DETER"
DIR_NASA = DIR_PREP / "NasaPower"
DIR_SISAM = DIR_PREP / "SISAM"
DIR_EDA = BASE / "datos" / "datasets_eda"

# ---------------------------------------------------------------------------
# 3. Constantes comunes (identicas en los 7 notebooks)
# ---------------------------------------------------------------------------
FECHA_INICIO_DATOS = "2022-01-01"
FECHA_FIN_EXTRACCION = "2026-11-01"      # DETER y BDQueimadas
PRIMERA_SEMANA = "2022-01-03"             # primer lunes con semana completa (2022-01-01 es sabado)
ULTIMA_SEMANA = "2026-10-19"              # lunes de la ultima semana completa (termina domingo 2026-10-25)
SATELITE_REFERENCIA = "AQUA_M-T"
ESTADOS_AMAZONIA_LEGAL = ["ACRE", "AMAPÁ", "AMAZONAS", "MARANHÃO", "MATO GROSSO", "PARÁ", "RONDÔNIA", "RORAIMA", "TOCANTINS"]
MAPA_UF = {"ACRE": "AC", "AMAPÁ": "AP", "AMAZONAS": "AM", "MARANHÃO": "MA", "MATO GROSSO": "MT", "PARÁ": "PA", "RONDÔNIA": "RO", "RORAIMA": "RR", "TOCANTINS": "TO"}
CORRECCIONES_MUNICIPIO = {"ELDORADO DOS CARAJAS": "ELDORADO DO CARAJAS", "SANTA ISABEL DO PARA": "SANTA IZABEL DO PARA"}
FECHA_CORTE_CENTROIDES = "2024-12-31"     # los centroides se fijan con focos hasta esta fecha

# ---------------------------------------------------------------------------
# 4. Funciones comunes
# ---------------------------------------------------------------------------
def normalizar_texto(texto):
    # mayusculas, sin tildes ni espacios sobrantes
    if pd.isna(texto):
        return texto
    texto = str(texto).upper().strip()
    return unicodedata.normalize("NFKD", texto).encode("ASCII", "ignore").decode("utf-8")


def semana_lunes(fecha):
    # lunes de la semana (lunes a domingo) de cada fecha; recibe una Serie de fechas
    f = pd.to_datetime(fecha).dt.normalize()
    return f - pd.to_timedelta(f.dt.day_of_week, unit="D")


SEMANAS = pd.date_range(PRIMERA_SEMANA, ULTIMA_SEMANA, freq="W-MON")   # calendario de semanas completas
assert len(SEMANAS) == 251, f"El calendario deberia tener 251 semanas y tiene {len(SEMANAS)}"
assert (SEMANAS.day_of_week == 0).all(), "Todas las semanas deben empezar en lunes"


def verificar_entradas(archivos):
    # imprime los archivos de entrada con su tamano y falla si falta alguno
    faltan = []
    for a in map(Path, archivos):
        if a.exists():
            print(f"  OK  {a.relative_to(BASE)}  ({a.stat().st_size / 1e6:,.1f} MB)")
        else:
            faltan.append(a)
    if faltan:
        lista = "\n".join(f"  - {a}" for a in faltan)
        raise FileNotFoundError("Faltan archivos de entrada (copialos a esas carpetas y vuelve a ejecutar):\n" + lista)


print("BASE:", BASE)
print("Calendario:", SEMANAS[0].date(), "a", SEMANAS[-1].date(), f"({len(SEMANAS)} semanas)")

ESPERADOS_SISAM = [DIR_SISAM / f"sisam_municipio_reanalise_diario_{a}-01-01_{a}-12-31.csv" for a in (2022, 2023, 2024, 2025)]
ESPERADOS_SISAM.append(DIR_SISAM / "sisam_municipio_forecast_diario_2026-01-01_2026-10-31.csv")
print("\nArchivos de entrada esperados:")
verificar_entradas(ESPERADOS_SISAM)

Mounted at /content/drive
BASE: /content/drive/MyDrive/BIODIVERSITY_GUARD
Calendario: 2022-01-03 a 2026-10-19 (251 semanas)

Archivos de entrada esperados:
  OK  datos/datasets_preprocesamiento/SISAM/sisam_municipio_reanalise_diario_2022-01-01_2022-12-31.csv  (22.2 MB)
  OK  datos/datasets_preprocesamiento/SISAM/sisam_municipio_reanalise_diario_2023-01-01_2023-12-31.csv  (22.2 MB)
  OK  datos/datasets_preprocesamiento/SISAM/sisam_municipio_reanalise_diario_2024-01-01_2024-12-31.csv  (22.3 MB)
  OK  datos/datasets_preprocesamiento/SISAM/sisam_municipio_reanalise_diario_2025-01-01_2025-12-31.csv  (21.9 MB)
  OK  datos/datasets_preprocesamiento/SISAM/sisam_municipio_forecast_diario_2026-01-01_2026-10-31.csv  (18.1 MB)


## Cabeceras reales y armonización

Cabeceras observadas en los archivos crudos:
- Reanálisis: `CD_MUN, municipio, estado, data, pm10_reanalise, pm2_5_reanalise, o3_reanalise, no2_reanalise, co_reanalise, so2_reanalise`.
- Pronóstico: `CD_MUN, estado, municipio, data, pm2_5_forecast_0, pm10_forecast_0, o3_forecast_0, co_forecast_0, no2_forecast_0, so2_forecast_0` (distinto orden de columnas).

El mapeo es inequívoco: cada variable del pronóstico se llama igual que la del reanálisis, cambiando el sufijo `_forecast_0` por `_reanalise`. El sufijo `_0` se interpreta como el pronóstico de plazo 0 (el día mismo); **es un supuesto a confirmar con la documentación de SISAM**. Si apareciera cualquier otra columna de pronóstico (otro plazo), el notebook se detiene en vez de adivinar.

In [2]:
VARIABLES_SISAM = ["pm10", "pm2_5", "o3", "no2", "co", "so2"]
COLS_REANALISE = [f"{v}_reanalise" for v in VARIABLES_SISAM]
RENOMBRAR_PRONOSTICO = {f"{v}_forecast_0": f"{v}_reanalise" for v in VARIABLES_SISAM}


def leer_sisam(ruta):
    d = pd.read_csv(ruta, encoding="utf-8-sig")
    tipo = "reanalise" if "_reanalise_" in ruta.name else "forecast" if "_forecast_" in ruta.name else None
    assert tipo, f"No se puede detectar el tipo (reanalise o forecast) por el nombre: {ruta.name}"
    if tipo == "forecast":
        extra = [c for c in d.columns if "_forecast_" in c and c not in RENOMBRAR_PRONOSTICO]
        assert not extra, f"Columnas de pronostico sin mapeo inequivoco en {ruta.name}: {extra}"
        faltan = set(RENOMBRAR_PRONOSTICO) - set(d.columns)
        assert not faltan, f"Faltan columnas de pronostico en {ruta.name}: {faltan}"
        d = d.rename(columns=RENOMBRAR_PRONOSTICO)
    base = ["CD_MUN", "municipio", "estado", "data"] + COLS_REANALISE
    assert set(base) <= set(d.columns), f"{ruta.name}: faltan columnas {set(base) - set(d.columns)}"
    d = d[base].copy()
    d["fuente_sisam"] = "reanalise" if tipo == "reanalise" else "pronostico"
    d["data"] = pd.to_datetime(d["data"], format="%d/%m/%Y", errors="coerce")   # dd/mm/aaaa tambien en el pronostico
    assert d["data"].notna().all(), f"{ruta.name}: {d['data'].isna().sum()} fechas con formato distinto de dd/mm/aaaa"
    return d


archivos = sorted(DIR_SISAM.glob("sisam_municipio_*.csv"))
assert set(ESPERADOS_SISAM) <= set(archivos), "Faltan archivos esperados de SISAM"
partes = []
for a in archivos:
    d = leer_sisam(a)
    partes.append(d)
    print(f"{a.name}: {len(d):,} filas | {d['CD_MUN'].nunique()} municipios | {d['data'].min().date()} a {d['data'].max().date()} | {d['fuente_sisam'].iloc[0]}")

# ambas fuentes quedaron con las mismas columnas
assert all(list(p.columns) == list(partes[0].columns) for p in partes), "Las columnas de las fuentes no coinciden tras armonizar"
df_sisam = pd.concat(partes, ignore_index=True)
print("\nTotal:", df_sisam.shape)

sisam_municipio_forecast_diario_2026-01-01_2026-10-31.csv: 245,936 filas | 809 municipios | 2026-01-01 a 2026-10-31 | pronostico
sisam_municipio_reanalise_diario_2022-01-01_2022-12-31.csv: 294,920 filas | 808 municipios | 2022-01-01 a 2022-12-31 | reanalise
sisam_municipio_reanalise_diario_2023-01-01_2023-12-31.csv: 294,920 filas | 808 municipios | 2023-01-01 a 2023-12-31 | reanalise
sisam_municipio_reanalise_diario_2024-01-01_2024-12-31.csv: 296,094 filas | 809 municipios | 2024-01-01 a 2024-12-31 | reanalise
sisam_municipio_reanalise_diario_2025-01-01_2025-12-31.csv: 295,285 filas | 809 municipios | 2025-01-01 a 2025-12-31 | reanalise

Total: (1427155, 11)


In [3]:
# SISAM escribe distinto a dos municipios (verificado en los archivos crudos):
# - TABOCAO (TO): BDQueimadas lo llama FORTALEZA DO TABOCAO (nombre anterior del municipio).
# - SANTO ANTONIO DO/DE LEVERGER (MT): el mismo codigo IBGE aparece con las dos grafias segun el archivo.
CORRECCIONES_SISAM = {"TABOCAO": "FORTALEZA DO TABOCAO", "SANTO ANTONIO DE LEVERGER": "SANTO ANTONIO DO LEVERGER"}

df_sisam["municipio_norm"] = df_sisam["municipio"].apply(normalizar_texto).replace(CORRECCIONES_SISAM)
df_sisam["uf"] = df_sisam["estado"].map(MAPA_UF)
assert df_sisam["uf"].notna().all(), f"Estados sin mapear: {df_sisam.loc[df_sisam['uf'].isna(), 'estado'].unique()}"

# cada codigo IBGE debe tener un unico (municipio_norm, uf) y viceversa
pares_cod = df_sisam[["CD_MUN", "municipio_norm", "uf"]].drop_duplicates()
assert pares_cod.groupby("CD_MUN").size().max() == 1, "Un codigo IBGE aparece con dos nombres distintos"
assert pares_cod.groupby(["municipio_norm", "uf"]).size().max() == 1, "Dos codigos IBGE comparten (municipio_norm, uf)"
print("Municipios unicos (codigo IBGE):", df_sisam["CD_MUN"].nunique())
print("Pares unicos (municipio_norm, uf):", len(pares_cod))

Municipios unicos (codigo IBGE): 809
Pares unicos (municipio_norm, uf): 809


## Duplicados

La clave `(municipio_norm, uf, data)` no debe repetirse (incluyendo el cruce entre reanálisis y pronóstico). Si se repite, el notebook se detiene mostrando ejemplos: **no se decide aquí cómo resolver** varias corridas por día.

In [4]:
clave_sisam = ["municipio_norm", "uf", "data"]
dup = df_sisam[df_sisam.duplicated(clave_sisam, keep=False)]
if len(dup):
    print(f"Se encontraron {len(dup):,} filas con clave repetida. Ejemplos:")
    print(dup.sort_values(clave_sisam).head(20).to_string(index=False))
    raise AssertionError("Hay duplicados en (municipio_norm, uf, data): decidir con el equipo como resolverlos antes de seguir")
print("OK: sin duplicados en (municipio_norm, uf, data)")

OK: sin duplicados en (municipio_norm, uf, data)


### Hallazgos de esta etapa

**Qué se observa**
- Se unieron **1,427,155** filas diarias: cuatro archivos de reanálisis (2022 a 2025) y uno de pronóstico (2026).
- Hay **809** municipios por código IBGE y **809** pares `(municipio_norm, uf)`, sin repetirse.
- No hay duplicados en `(municipio_norm, uf, data)`.

**Por qué se decidió así**
- Se corrigieron dos nombres solo en este notebook: `TABOCAO` pasa a `FORTALEZA DO TABOCAO` y `SANTO ANTONIO DE LEVERGER` pasa a `SANTO ANTONIO DO LEVERGER`.
- La primera es el nombre anterior del municipio en BDQueimadas. En la segunda, el mismo código IBGE aparecía con dos grafías según el archivo.
- Con las correcciones, cada código IBGE tiene un solo nombre y viceversa.


## Cobertura por año y por fuente

Municipios, días, rango de fechas y nulos por variable. Los niveles del pronóstico y del reanálisis no son comparables en todas las variables (son productos distintos); por eso `fuente_sisam` se conserva hasta el dataset final.

In [5]:
df_sisam["anio"] = df_sisam["data"].dt.year
cobertura = df_sisam.groupby(["anio", "fuente_sisam"]).agg(
    filas=("data", "size"), municipios=("municipio_norm", "nunique"), dias=("data", "nunique"),
    desde=("data", "min"), hasta=("data", "max"),
)
print(cobertura)
print("\nNulos por variable y anio:")
print(df_sisam.groupby("anio")[COLS_REANALISE].apply(lambda g: g.isna().sum()))
print("\nMedia por variable y fuente (para ver diferencias de nivel):")
print(df_sisam.groupby("fuente_sisam")[COLS_REANALISE].mean().round(2))

                    filas  municipios  dias      desde      hasta
anio fuente_sisam                                                
2022 reanalise     294920         803   365 2022-01-01 2022-12-31
2023 reanalise     294920         803   365 2023-01-01 2023-12-31
2024 reanalise     296094         804   366 2024-01-01 2024-12-31
2025 reanalise     295285         804   365 2025-01-01 2025-12-31
2026 pronostico    245936         804   304 2026-01-01 2026-10-31

Nulos por variable y anio:
      pm10_reanalise  pm2_5_reanalise  o3_reanalise  no2_reanalise  \
anio                                                                 
2022               0                0             0              0   
2023               0                0             0              0   
2024               0                0             0              0   
2025               0                0             0              0   
2026               0                0             0              0   

      co_reanalise 

### Hallazgos de esta etapa

**Qué se observa**
- 2022 a 2025 son reanálisis (**365**, **365**, **366** y **365** días). 2026 es pronóstico, con **304** días (1 de enero a 31 de octubre).
- No hay nulos en ninguna variable ni año.
- Los niveles cambian entre fuentes (media reanálisis frente a pronóstico): PM10 **22.18** frente a **8.82**, PM2.5 **16.10** frente a **7.58**, O3 **31.31** frente a **45.79** y NO2 **1.32** frente a **2.62**. CO (**0.18** y **0.15**) y SO2 (**0.43** y **0.42**) casi no cambian.
- SISAM cubre **809** municipios y los **543** del panel están todos.

**Por qué se decidió así**
- Reanálisis y pronóstico son productos distintos. Por eso se conserva la columna `fuente_sisam` hasta el dataset final y no se corrige el nivel.

**Qué implica para el modelado**
- **Limitación:** el sufijo `_forecast_0` se interpretó como pronóstico de plazo 0. Falta confirmarlo con la documentación de SISAM.
- **Limitación:** 2026 usa pronóstico sin calibrar. Su salto de nivel puede confundirse con un efecto real si el modelo no conoce la fuente.


## Cobertura de municipios frente al panel

Compara los pares `(municipio_norm, uf)` de SISAM con los del panel. Si todavía no existe `panel_semanal_base.csv` (el notebook 03 se ejecuta después), se usa `centroides_municipio.csv` (municipios con focos).

In [6]:
ruta_panel = DIR_PREP / "panel_semanal_base.csv"
ruta_cent = DIR_BDQ / "centroides_municipio.csv"
if ruta_panel.exists():
    ref = pd.read_csv(ruta_panel, usecols=["municipio_norm", "uf"]).drop_duplicates()
    nombre_ref = "panel_semanal_base.csv"
elif ruta_cent.exists():
    ref = pd.read_csv(ruta_cent, usecols=["municipio_norm", "uf"]).drop_duplicates()
    nombre_ref = "centroides_municipio.csv (solo municipios con focos)"
else:
    ref = None

if ref is None:
    print("No hay panel ni centroides para comparar: se omite la verificacion.")
else:
    p_ref = set(map(tuple, ref.values))
    p_sis = set(map(tuple, df_sisam[["municipio_norm", "uf"]].drop_duplicates().values))
    print(f"Referencia: {nombre_ref}")
    print(f"Pares en la referencia: {len(p_ref)} | en SISAM: {len(p_sis)} | en ambos: {len(p_ref & p_sis)}")
    print("En la referencia pero NO en SISAM:", sorted(p_ref - p_sis))

Referencia: panel_semanal_base.csv
Pares en la referencia: 543 | en SISAM: 809 | en ambos: 543
En la referencia pero NO en SISAM: []


In [7]:
columnas_finales = ["data", "municipio_norm", "estado", "uf"] + COLS_REANALISE + ["fuente_sisam"]
df_limpio = df_sisam[columnas_finales].sort_values(["municipio_norm", "uf", "data"]).reset_index(drop=True)

df_limpio.to_csv(DIR_SISAM / "sisam_limpio.csv", index=False, encoding="utf-8", date_format="%Y-%m-%d")
print("Guardado:", (DIR_SISAM / "sisam_limpio.csv").relative_to(BASE), df_limpio.shape)
print(df_limpio.dtypes)

Guardado: datos/datasets_preprocesamiento/SISAM/sisam_limpio.csv (1427155, 11)
data               datetime64[ns]
municipio_norm             object
estado                     object
uf                         object
pm10_reanalise            float64
pm2_5_reanalise           float64
o3_reanalise              float64
no2_reanalise             float64
co_reanalise              float64
so2_reanalise             float64
fuente_sisam               object
dtype: object
